# Première lecture des documents FOMC

Point de départ : `data/raw/fomc_documents_raw.csv`.

Le fichier contient les textes extraits du site de la Fed. Les transformations ci-dessous sont faites dans le notebook, pas dans un CSV préparé à l'avance.

In [ ]:
import os
import re
import tempfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "fomc_nlp_matplotlib"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "fomc_documents_raw.csv").exists():
    ROOT = ROOT.parent

raw_path = ROOT / "data" / "raw" / "fomc_documents_raw.csv"
raw = pd.read_csv(raw_path, parse_dates=["date"])
raw.head(2)

## 1. Contrôle rapide

On vérifie d'abord que les dates sont uniques et que les deux textes sont présents.

In [ ]:
pd.Series({
    "rows": len(raw),
    "first_date": raw["date"].min().date(),
    "last_date": raw["date"].max().date(),
    "duplicated_dates": raw["date"].duplicated().sum(),
    "empty_statements": raw["statement_text"].fillna("").str.len().eq(0).sum(),
    "empty_minutes": raw["minutes_text"].fillna("").str.len().eq(0).sum(),
})

In [ ]:
raw[["date", "statement_url", "minutes_url"]].head()

## 2. Nettoyage minimal

Nettoyage volontairement simple : minuscules, espaces propres, comptage de mots par regex.

In [ ]:
TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)?", re.IGNORECASE)

def clean_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.replace(" ", " ")
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def count_words(text):
    return len(TOKEN_RE.findall(clean_text(text)))

work = raw.copy()
work["statement_clean_text"] = work["statement_text"].map(clean_text)
work["minutes_clean_text"] = work["minutes_text"].map(clean_text)
work["statement_n_words"] = work["statement_text"].map(count_words)
work["minutes_n_words"] = work["minutes_text"].map(count_words)
work[["date", "statement_n_words", "minutes_n_words"]].head()

In [ ]:
lengths = pd.concat([
    work[["date", "year", "statement_n_words"]].rename(columns={"statement_n_words": "n_words"}).assign(document_type="statement"),
    work[["date", "year", "minutes_n_words"]].rename(columns={"minutes_n_words": "n_words"}).assign(document_type="minutes"),
], ignore_index=True)

lengths.groupby("document_type")["n_words"].describe().round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(data=lengths, x="date", y="n_words", hue="document_type", ax=ax)
ax.set_yscale("log")
ax.set_title("Document length")
ax.set_xlabel("Meeting date")
ax.set_ylabel("Words, log scale");

## 3. Décision de taux

Label simple, extrait du Statement. C'est utile pour lire les graphiques, mais ce n'est pas une vérité externe importée.

In [ ]:
RATE_PATTERNS = {
    "hike": [
        r"\b(decided|voted|agreed|approved)\b.{0,80}\b(raise|raising|increase|increasing)\b.{0,80}\b(target|federal funds|discount rate)\b",
        r"\b(raise|raising|increase|increasing|increased)\b.{0,80}\b(target range|target for the federal funds rate|federal funds rate)\b",
    ],
    "cut": [
        r"\b(decided|voted|agreed|approved)\b.{0,80}\b(lower|lowering|reduce|reducing|decrease|decreasing|cut)\b.{0,80}\b(target|federal funds|discount rate)\b",
        r"\b(lower|lowering|reduce|reducing|decrease|decreasing|cut)\b.{0,80}\b(target range|target for the federal funds rate|federal funds rate)\b",
    ],
    "hold": [
        r"\b(decided|voted|agreed)\b.{0,80}\b(maintain|keep|leave)\b.{0,80}\b(target range|target for the federal funds rate|federal funds rate)\b",
        r"\b(will|would|shall|to)\b.{0,20}\b(maintain|keep|leave)\b.{0,80}\b(target range|target for the federal funds rate|federal funds rate)\b",
        r"\b(maintain|maintaining|keep|keeping|kept|leave|leaving)\b.{0,80}\b(target range|target for the federal funds rate|federal funds rate)\b",
        r"\b(target range|target for the federal funds rate|federal funds rate)\b.{0,80}\b(unchanged|maintained)\b",
    ],
}

def infer_rate_decision(text):
    text = clean_text(text)
    for label in ["hike", "cut", "hold"]:
        if any(re.search(pattern, text, flags=re.DOTALL) for pattern in RATE_PATTERNS[label]):
            return label
    return "unknown"

work["rate_decision"] = work["statement_text"].map(infer_rate_decision)
work["rate_decision"].value_counts(dropna=False)

## 4. Premiers dictionnaires

Dictionnaires courts, faciles à modifier. Les scores sont divisés par le nombre de mots.

In [ ]:
HAWKISH_TERMS = [
    "inflation", "price stability", "elevated", "tightening", "restrictive", "firming",
    "increase", "increased", "increases", "pressure", "pressures", "overheating",
    "strong labor market", "higher rates", "persistent inflation", "inflationary pressures",
]

DOVISH_TERMS = [
    "unemployment", "slowdown", "weakness", "accommodative", "support", "downside risks",
    "lower rates", "easing", "recession", "recovery", "labor market slack", "weak demand",
    "economic weakness",
]

UNCERTAINTY_TERMS = [
    "uncertainty", "uncertain", "risks", "risk", "downside", "upside", "volatility",
    "financial conditions", "stress", "disruptions", "concerns", "monitor", "likely", "may", "could",
]

def term_count(text, terms):
    text = clean_text(text)
    total = 0
    for term in terms:
        pattern = re.escape(term.lower()).replace(r"\ ", r"\s+")
        total += len(re.findall(rf"(?<![a-z]){pattern}(?![a-z])", text))
    return total

def add_dictionary_scores(df, corpus):
    text_col = f"{corpus}_clean_text"
    n_words = df[f"{corpus}_n_words"].replace(0, np.nan)
    hawkish = df[text_col].map(lambda x: term_count(x, HAWKISH_TERMS))
    dovish = df[text_col].map(lambda x: term_count(x, DOVISH_TERMS))
    uncertainty = df[text_col].map(lambda x: term_count(x, UNCERTAINTY_TERMS))
    df[f"{corpus}_hawkish_score"] = (hawkish / n_words).fillna(0)
    df[f"{corpus}_dovish_score"] = (dovish / n_words).fillna(0)
    df[f"{corpus}_tone_score"] = df[f"{corpus}_hawkish_score"] - df[f"{corpus}_dovish_score"]
    df[f"{corpus}_uncertainty_score"] = (uncertainty / n_words).fillna(0)

add_dictionary_scores(work, "statement")
add_dictionary_scores(work, "minutes")

In [ ]:
tone = pd.concat([
    work[["date", "rate_decision", "statement_tone_score"]].rename(columns={"statement_tone_score": "tone_score"}).assign(document_type="statement"),
    work[["date", "rate_decision", "minutes_tone_score"]].rename(columns={"minutes_tone_score": "tone_score"}).assign(document_type="minutes"),
], ignore_index=True)

fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(data=tone, x="date", y="tone_score", hue="document_type", ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Hawkish minus dovish score")
ax.set_xlabel("Meeting date")
ax.set_ylabel("Score");

In [ ]:
tone.groupby(["document_type", "rate_decision"])["tone_score"].mean().unstack().round(5)

In [ ]:
uncertainty = pd.concat([
    work[["date", "statement_uncertainty_score"]].rename(columns={"statement_uncertainty_score": "uncertainty_score"}).assign(document_type="statement"),
    work[["date", "minutes_uncertainty_score"]].rename(columns={"minutes_uncertainty_score": "uncertainty_score"}).assign(document_type="minutes"),
], ignore_index=True)

fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(data=uncertainty, x="date", y="uncertainty_score", hue="document_type", ax=ax)
ax.set_title("Uncertainty score")
ax.set_xlabel("Meeting date")
ax.set_ylabel("Terms per word");

## 5. Similarité TF-IDF

On calcule deux choses : changement d'un document au suivant, puis distance entre Statement et Minutes pour une même réunion.

In [ ]:
def successive_shift(df, text_col):
    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=5000)
    matrix = vectorizer.fit_transform(df[text_col].fillna(""))
    shifts = [np.nan]
    for i in range(1, matrix.shape[0]):
        sim = cosine_similarity(matrix[i], matrix[i - 1])[0, 0]
        shifts.append(1 - sim)
    return shifts

work = work.sort_values("date").reset_index(drop=True)
work["statement_shift_tfidf"] = successive_shift(work, "statement_clean_text")
work["minutes_shift_tfidf"] = successive_shift(work, "minutes_clean_text")

In [ ]:
shift = pd.concat([
    work[["date", "rate_decision", "statement_shift_tfidf"]].rename(columns={"statement_shift_tfidf": "shift_tfidf"}).assign(document_type="statement"),
    work[["date", "rate_decision", "minutes_shift_tfidf"]].rename(columns={"minutes_shift_tfidf": "shift_tfidf"}).assign(document_type="minutes"),
], ignore_index=True)

fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(data=shift, x="date", y="shift_tfidf", hue="document_type", ax=ax)
ax.set_title("TF-IDF shift from previous document")
ax.set_xlabel("Meeting date")
ax.set_ylabel("1 - cosine similarity");

In [ ]:
top_statement_shifts = work.nlargest(10, "statement_shift_tfidf")[["date", "rate_decision", "statement_shift_tfidf", "statement_tone_score", "statement_uncertainty_score", "statement_text"]]
top_minutes_shifts = work.nlargest(10, "minutes_shift_tfidf")[["date", "rate_decision", "minutes_shift_tfidf", "minutes_tone_score", "minutes_uncertainty_score", "minutes_text"]]

top_statement_shifts.assign(statement_text=lambda x: x["statement_text"].str.slice(0, 300))

In [ ]:
texts = work["statement_clean_text"].fillna("").tolist() + work["minutes_clean_text"].fillna("").tolist()
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=5000)
matrix = vectorizer.fit_transform(texts)
n = len(work)

same_meeting_similarity = []
for i in range(n):
    same_meeting_similarity.append(cosine_similarity(matrix[i], matrix[i + n])[0, 0])

work["same_meeting_distance_tfidf"] = 1 - np.array(same_meeting_similarity)
work["same_meeting_distance_tfidf"].describe().round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(data=work, x="date", y="same_meeting_distance_tfidf", ax=ax)
ax.set_title("Statement-Minutes distance")
ax.set_xlabel("Meeting date")
ax.set_ylabel("1 - cosine similarity");

## 6. Mots fréquents

Simple comptage pour voir les termes dominants dans chaque corpus.

In [ ]:
def top_terms(texts, n=25):
    vectorizer = CountVectorizer(stop_words="english", ngram_range=(1, 2), max_features=10000)
    matrix = vectorizer.fit_transform(texts)
    counts = np.asarray(matrix.sum(axis=0)).ravel()
    terms = vectorizer.get_feature_names_out()
    idx = counts.argsort()[::-1][:n]
    return pd.DataFrame({"term": terms[idx], "count": counts[idx]})

top_statement_terms = top_terms(work["statement_clean_text"])
top_minutes_terms = top_terms(work["minutes_clean_text"])
top_statement_terms.head(15)

## Notes pour la suite

- Les Minutes sont beaucoup plus longues : comparer les scores normalisés, pas les comptes bruts.
- Les labels de décision sont inférés dans le notebook, donc à relire si on les utilise dans le rapport.
- Les pics TF-IDF doivent être lus qualitativement, date par date.